# 04 — Writing Files & Round-Trips

Every reader has a mirror-image writer, and the writers regenerate every
file section **from the parsed DataFrames** — no raw text is copied
through. That makes `read → edit a DataFrame → write` a first-class
workflow, verified end-to-end: the sample model reproduces baseline
heads *exactly* through the real IWFM executables after a full
read/write round-trip of all its inputs.

This notebook shows:

- simple file round-trips (nodes, elements, stratigraphy)
- editing time series (an ET climate scenario)
- editing model parameters (GW initial heads) and writing back
- the two IWFM quirks the writers handle for you

**Requires:** the sample model. All writes go to a temporary directory.

In [1]:
import os
import tempfile
from pathlib import Path


def find_sample_model():
    env = os.environ.get("IWFM_SAMPLE_MODEL")
    if env:
        return Path(env)
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / ".assets" / "sample_model"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("sample model not found — see notebook 01")


SAMPLE_MODEL = find_sample_model()
PP_DIR = SAMPLE_MODEL / "Preprocessor"
SIM_DIR = SAMPLE_MODEL / "Simulation"
GW_DIR = SIM_DIR / "GW"

tmp = Path(tempfile.mkdtemp(prefix="iwfm_nb04_"))
print("writing to", tmp)

writing to C:\Users\serca\AppData\Local\Temp\iwfm_nb04_c_vfpkm7


## Round-trip: write, read back, compare

The basic contract: `write_*(read_*(path), out)` produces a file that
reads back identically.

In [2]:
import numpy as np

from iwfm_io import read_nodes, write_nodes

original = read_nodes(PP_DIR / "NodeXY.dat")
write_nodes(original, tmp / "NodeXY_copy.dat")
copy = read_nodes(tmp / "NodeXY_copy.dat")

assert len(copy.data) == len(original.data)
assert np.allclose(copy.data[["x", "y"]], original.data[["x", "y"]])
print(f"nodes round-trip OK ({len(copy.data)} nodes, coordinates identical)")

nodes round-trip OK (441 nodes, coordinates identical)


In [3]:
from iwfm_io import read_strata, write_strata

strata = read_strata(PP_DIR / "Strata.dat")
write_strata(strata, tmp / "Strata_copy.dat")
back = read_strata(tmp / "Strata_copy.dat")

num = strata.data.select_dtypes("number").columns
max_diff = np.abs(strata.data[num].values - back.data[num].values).max()
print(f"stratigraphy round-trip OK ({back.n_layers} layers, "
      f"max numeric diff = {max_diff:.2e})")

stratigraphy round-trip OK (2 layers, max numeric diff = 0.00e+00)


## Edit a time series: −20 % ET climate scenario

Time-series files carry their data as a DataFrame (`date` + one column
per site). Scale the values, write, verify.

In [4]:
import copy as _copy

from iwfm_io import read_et, write_et

et = read_et(SIM_DIR / "ET.dat")
value_cols = [c for c in et.data.columns if c != "date"]
print(f"ET: {len(et.data)} timesteps x {len(value_cols)} columns, "
      f"mean {et.data[value_cols[0]].mean():.4f}")

scenario = _copy.deepcopy(et)
scenario.data[value_cols] = et.data[value_cols] * 0.80
write_et(scenario, tmp / "ET_dry.dat")

verify = read_et(tmp / "ET_dry.dat")
ratio = verify.data[value_cols[0]].mean() / et.data[value_cols[0]].mean()
print(f"written scenario mean ratio = {ratio:.4f} (expect 0.80)")

ET: 12 timesteps x 7 columns, mean 4.1000
written scenario mean ratio = 0.8000 (expect 0.80)


## Edit model parameters: raise layer-1 initial heads

The GW main file regenerates entirely from its DataFrames — aquifer
parameters, anomalies, hydrograph specs, initial heads, everything.

One argument matters here: **`base_dir`**. IWFM rejects absolute paths
in its input files (it prefixes `.\`), so the component-main writers
take the simulation working directory and relativise every referenced
path against it.

In [5]:
from iwfm_io import read_gw_main, write_gw_main

gw = read_gw_main(GW_DIR / "GW_MAIN.dat")
before = gw.initial_heads["head_layer_1"].iloc[0]

gw.initial_heads["head_layer_1"] += 5.0
write_gw_main(gw, tmp / "GW_MAIN_modified.dat", base_dir=SIM_DIR)

gw2 = read_gw_main(tmp / "GW_MAIN_modified.dat")
after = gw2.initial_heads["head_layer_1"].iloc[0]
print(f"initial head at node 1: {before} -> {after}")
assert after == before + 5.0

initial head at node 1: 280.0 -> 285.0


In [6]:
# The regenerated file is a normal IWFM input — comments, keyed values,
# tables. Here is its head:
print("\n".join((tmp / "GW_MAIN_modified.dat").read_text().splitlines()[:15]))

#4.0
C***  DO NOT DELETE ABOVE LINE ***
C
C*******************************************************************************
C
C                  INTEGRATED WATER FLOW MODEL (IWFM)
C
C*******************************************************************************
C
C                  GROUNDWATER COMPONENT MAIN DATA FILE
C                        Groundwater Component
C                         *** Version 4.0 ***
C
C
C             Project:  IWFM Public Release


## The whole preprocessor tree

In [7]:
from iwfm_io import read_elements, read_preprocessor, write_elements, write_preprocessor

pp = read_preprocessor(PP_DIR / "PreProcessor_MAIN.IN")
out_dir = tmp / "preprocessor_copy"
out_dir.mkdir()

write_preprocessor(pp, out_dir / "PreProcessor_MAIN.IN", base_dir=out_dir)
write_nodes(pp.children["node"], out_dir / "NodeXY.dat")
write_elements(pp.children["element"], out_dir / "Element.dat")

n = len(read_elements(out_dir / "Element.dat").data)
print(f"preprocessor tree copied: main + {n} elements + "
      f"{len(read_nodes(out_dir / 'NodeXY.dat').data)} nodes")

preprocessor tree copied: main + 400 elements + 441 nodes


## Two load-bearing IWFM quirks (handled for you)

Worth knowing because they are invisible until a hand-rolled writer
breaks a model:

1. **Terminating comment lines.** IWFM reads the parametric-grid node
   list and the root-zone sub-file list as "data lines until a comment
   line" — the writers emit a terminating comment line after both.
2. **No absolute paths.** IWFM prefixes referenced paths with `.\`, so
   absolute paths break it. Writers take `base_dir` and keep every
   reference relative (see the GW main example above).

And a safety property: writers flush through a temp file + atomic
`os.replace`, so a crash mid-write never leaves a half-written input.

## Proof it works: the executable round-trip test

`tests/io/test_exe_roundtrip.py` regenerates *every* input of the sample
model through the writers, runs the real PreProcessor + Simulation
executables on the result, and asserts the simulated heads match the
baseline **exactly**. Run it yourself (Windows, ~3 min):

```
set IWFM_RUN_EXE_TESTS=1 && pytest tests/io/test_exe_roundtrip.py
```

For scripted model edits you usually don't call writers directly —
notebook 06 shows `create_scenario()` + change factories, which wrap
this machinery.

In [8]:
import shutil

shutil.rmtree(tmp)
print("temporary directory removed")

temporary directory removed
